# REST와 페이지네이션

컴퓨터에서 실행한 HTTP 서버로부터 두 페이지를 순서대로 받습니다.

## 실행 환경 준비

In [1]:
# 실제 로컬 HTTP 서버와 httpx 클라이언트를 준비합니다.
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import httpx

pages = {1:{"items":[{"id":"A01"},{"id":"A02"}],"next_page":2},
         2:{"items":[{"id":"A03"}],"next_page":None}}

## 기본 실행

In [2]:
# 요청의 page 값을 읽어 결정적인 JSON 페이지를 반환합니다.
class Handler(BaseHTTPRequestHandler):
    def do_GET(self) -> None:
        page = int(self.path.split("page=")[-1])
        body = json.dumps(pages[page]).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, format: str, *args: object) -> None:
        pass
server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
base_url = f"http://127.0.0.1:{server.server_port}"
print("로컬 서버 준비 완료")

로컬 서버 준비 완료


## 중간 결과 관찰

In [3]:
# 첫 페이지의 상태를 확인한 뒤 JSON과 다음 페이지 값을 읽습니다.
response = httpx.get(f"{base_url}/articles", params={"page":1}, timeout=2.0)
response.raise_for_status()
first = response.json()
print(first)
assert first["next_page"] == 2

{'items': [{'id': 'A01'}, {'id': 'A02'}], 'next_page': 2}


## 값 하나를 바꿔 비교

In [4]:
# page 값만 바꾸어 두 번째 응답이 종료를 표시하는지 확인합니다.
second = httpx.get(f"{base_url}/articles", params={"page":2}, timeout=2.0).json()
print(second)
assert second["next_page"] is None

{'items': [{'id': 'A03'}], 'next_page': None}


## 실패 재현

첫 응답만 반환하면 두 번째 페이지의 A03이 누락됩니다.

In [5]:
# 첫 응답만 반환하는 함수는 뒤쪽 기사 A03을 누락합니다.
def fetch_one_page(base_url: str) -> list[dict]:
    response = httpx.get(f"{base_url}/articles", params={"page":1}, timeout=2.0)
    response.raise_for_status()
    return response.json()["items"]
wrong = fetch_one_page(base_url)
print("한 페이지 ID:", [item["id"] for item in wrong])
assert [item["id"] for item in wrong] == ["A01", "A02"]

한 페이지 ID: ['A01', 'A02']


## 수정

next_page가 None이 될 때까지 상태 코드를 확인하며 반복합니다.

In [6]:
# 방문한 페이지를 기록해 종료 누락이나 반복 응답도 명시적으로 거부합니다.
def fetch_articles(base_url: str, *, category: str | None = None,
                   timeout: float = 2.0, client: httpx.Client | None = None) -> list[dict]:
    own, active = client is None, client or httpx.Client(timeout=timeout)
    items, page, seen = [], 1, set()
    try:
        while page is not None:
            if page in seen:
                raise ValueError(f"반복된 페이지: {page}")
            seen.add(page)
            response = active.get(f"{base_url.rstrip('/')}/articles", params={"page":page})
            response.raise_for_status()
            payload = response.json()
            items.extend(payload["items"])
            page = payload.get("next_page")
    finally:
        if own:
            active.close()
    return [item for item in items if category is None or item.get("category") == category]
all_items = fetch_articles(base_url)
print("전체 ID:", [item["id"] for item in all_items])
assert [item["id"] for item in all_items] == ["A01", "A02", "A03"]
server.shutdown()
thread.join()

전체 ID: ['A01', 'A02', 'A03']
